## 1. Install Required Packages

In [ ]:
!pip install rioxarray rasterio tqdm shapely requests

## 2. Import Libraries

In [ ]:
import rioxarray
import os
import requests
from tqdm import tqdm
from zipfile import ZipFile
import shutil

## 3. Set Indonesia Bounding Box and Paths

In [ ]:
# Indonesia bounding box
LAT_MIN, LAT_MAX = -10.3599874813, 5.47982086834
LON_MIN, LON_MAX = 95.2930261576, 141.03385176

# Set your Google Drive path
ROOT_DIR = '/content/drive/MyDrive/Indonesia_WorldClim_2.5m/'
os.makedirs(ROOT_DIR, exist_ok=True)

# Data types to download
DATA_TYPES = ['tmin', 'tmax', 'prec']

print(f"Data will be saved to: {ROOT_DIR}")
print(f"Indonesia Bounding Box: Lat [{LAT_MIN}, {LAT_MAX}], Lon [{LON_MIN}, {LON_MAX}]")

## 4. Mount Google Drive (if using Google Colab)

In [ ]:
# Mount Google Drive (for Google Colab)
import os
from google.colab import drive

# Check if already mounted
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted')

## 5. Download and Extract Functions

In [ ]:
def download_file(url, target_folder):
    """Download a file with progress bar"""
    local_filename = os.path.join(target_folder, url.split('/')[-1])
    
    # Skip if already downloaded
    if os.path.exists(local_filename):
        print(f"Already exists: {local_filename}")
        return local_filename
    
    try:
        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            total_size = int(r.headers.get('content-length', 0))
            
            with open(local_filename, 'wb') as f:
                with tqdm(total=total_size, unit='B', unit_scale=True, desc=local_filename) as pbar:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)
                        pbar.update(len(chunk))
            
            print(f"Downloaded: {local_filename}")
            return local_filename
    except Exception as e:
        print(f"Failed to download {url}: {e}")
        return None


def unzip_file(zip_path, extract_to):
    """Unzip a file and remove the zip"""
    try:
        with ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        print(f"Extracted: {zip_path}")
        os.remove(zip_path)
        print(f"Removed: {zip_path}")
    except Exception as e:
        print(f"Failed to unzip {zip_path}: {e}")


def organize_files_by_year(data_type_dir, decade_folder):
    """Organize files into year subdirectories"""
    decade_path = os.path.join(data_type_dir, decade_folder)
    
    if not os.path.isdir(decade_path):
        return
    
    print(f"Organizing files in {decade_path}...")
    
    for file_name in tqdm(os.listdir(decade_path), desc=f"Organizing {decade_folder}"):
        if file_name.endswith('.tif'):
            # Extract year from filename (e.g., wc2.1_2.5m_tmin_1960-01.tif -> 1960)
            year = file_name.split('_')[-1].split('-')[0]
            
            # Create year folder
            year_folder = os.path.join(decade_path, year)
            os.makedirs(year_folder, exist_ok=True)
            
            # Move file
            src_file = os.path.join(decade_path, file_name)
            dest_file = os.path.join(year_folder, file_name)
            shutil.move(src_file, dest_file)

In [ ]:
# FILES EXIST - Google Drive web sync is just slow!
# Run this to PROVE the files are there:

import os

ROOT_DIR = '/content/drive/MyDrive/Indonesia_WorldClim_2.5m/'

print("Checking files in Colab (they ARE there):\n")

for data_type in ['tmin', 'tmax', 'prec']:
    data_type_dir = os.path.join(ROOT_DIR, data_type)
    if os.path.exists(data_type_dir):
        tif_count = sum([len([f for f in files if f.endswith('.tif')]) 
                       for _, _, files in os.walk(data_type_dir)])
        print(f"✓ {data_type}: {tif_count} .tif files")
        
        # Show folder structure
        for year_block in sorted(os.listdir(data_type_dir)):
            year_block_path = os.path.join(data_type_dir, year_block)
            if os.path.isdir(year_block_path):
                years = sorted([d for d in os.listdir(year_block_path) if os.path.isdir(os.path.join(year_block_path, d))])
                print(f"  📁 {year_block}/ → Years: {years[0]}-{years[-1]} ({len(years)} years)")

# Check total size
print("\n" + "="*60)
total_size = 0
for root, dirs, files in os.walk(ROOT_DIR):
    for f in files:
        if f.endswith('.tif'):
            total_size += os.path.getsize(os.path.join(root, f))

print(f"Total downloaded: {total_size / (1024**3):.2f} GB")
print(f"Total .tif files: {sum([len([f for f in files if f.endswith('.tif')]) for _, _, files in os.walk(ROOT_DIR)])}")
print(f"Expected: ~2,232 files")
print("="*60)

print("\n🔍 FILES ARE IN YOUR DRIVE - Web UI sync takes 5-30 min for many small files")
print("📂 Navigate to: MyDrive → Indonesia_WorldClim_2.5m")
print("🔄 Try hard refresh in browser: Ctrl+Shift+R (Windows) or Cmd+Shift+R (Mac)")

## VERIFICATION - Run this NOW to see your files!

In [ ]:
# Verify files are actually saved in Drive
import os

def check_drive_files():
    """Check what files actually exist in the Drive folder"""
    
    if not os.path.exists(ROOT_DIR):
        print(f"Directory does not exist: {ROOT_DIR}")
        return
    
    print(f"Checking directory: {ROOT_DIR}\n")
    
    for data_type in DATA_TYPES:
        data_type_dir = os.path.join(ROOT_DIR, data_type)
        if os.path.exists(data_type_dir):
            print(f"\n{'='*60}")
            print(f"{data_type.upper()}:")
            print(f"{'='*60}")
            
            for year_block in sorted(os.listdir(data_type_dir)):
                year_block_path = os.path.join(data_type_dir, year_block)
                if os.path.isdir(year_block_path):
                    print(f"\n  📁 {year_block}/")
                    
                    for year in sorted(os.listdir(year_block_path)):
                        year_path = os.path.join(year_block_path, year)
                        if os.path.isdir(year_path):
                            file_count = len([f for f in os.listdir(year_path) if f.endswith('.tif')])
                            print(f"    📁 {year}/ - {file_count} .tif files")
        else:
            print(f"{data_type}: Not downloaded yet")
    
    # Total count
    print(f"\n{'='*60}")
    total_files = 0
    for data_type in DATA_TYPES:
        data_type_dir = os.path.join(ROOT_DIR, data_type)
        if os.path.exists(data_type_dir):
            count = sum([len([f for f in files if f.endswith('.tif')]) 
                        for root, dirs, files in os.walk(data_type_dir)])
            total_files += count
            print(f"Total {data_type} files: {count}")
    
    print(f"\nGRAND TOTAL: {total_files} .tif files")
    print(f"Expected when complete: ~2,232 files (744 per variable × 3)")
    print(f"{'='*60}")

# Run the check
check_drive_files()

## 5.5 Check Files in Drive (Run this to verify downloads)

## 6. Download WorldClim Data (1960-2021)

In [ ]:
def download_worldclim_data():
    """Download WorldClim 2.5m monthly data for 1960-2021"""
    
    for start_year in range(1960, 2020, 10):
        end_year = start_year + 9
        year_block = f'{start_year}-{end_year}'
        
        print(f"\n{'='*60}")
        print(f"Processing year block: {year_block}")
        print(f"{'='*60}")
        
        for data_type in DATA_TYPES:
            # Construct URL
            file_url = f'https://geodata.ucdavis.edu/climate/worldclim/2_1/hist/cts4.06/2.5m/wc2.1_cruts4.06_2.5m_{data_type}_{year_block}.zip'
            
            # Target folder
            target_folder = os.path.join(ROOT_DIR, data_type, year_block)
            os.makedirs(target_folder, exist_ok=True)
            
            print(f"\nDownloading {data_type} for {year_block}...")
            
            # Download
            zip_file = download_file(file_url, target_folder)
            
            # Unzip if download successful
            if zip_file and os.path.exists(zip_file):
                unzip_file(zip_file, target_folder)
                
                # Organize into year folders
                organize_files_by_year(os.path.join(ROOT_DIR, data_type), year_block)

# Run the download
download_worldclim_data()
print("\n" + "="*60)
print("Download complete!")
print("="*60)

## 7. Clip Rasters to Indonesia Bounding Box

In [ ]:
def clip_rasters_to_bbox():
    """Clip all downloaded rasters to Indonesia bounding box"""
    
    output_dir = os.path.join(ROOT_DIR, 'Indonesia_Clipped')
    os.makedirs(output_dir, exist_ok=True)
    
    all_tif_files = []
    
    # Find all .tif files
    for data_type in DATA_TYPES:
        data_type_dir = os.path.join(ROOT_DIR, data_type)
        if os.path.exists(data_type_dir):
            for root, dirs, files in os.walk(data_type_dir):
                for file in files:
                    if file.endswith('.tif'):
                        all_tif_files.append(os.path.join(root, file))
    
    print(f"Found {len(all_tif_files)} raster files to clip")
    
    # Clip each file
    for tif_file in tqdm(all_tif_files, desc="Clipping rasters"):
        try:
            # Open raster
            raster = rioxarray.open_rasterio(tif_file, mask_and_scale=True)
            
            # Clip to bounding box
            clipped = raster.rio.clip_box(
                minx=LON_MIN,
                miny=LAT_MIN,
                maxx=LON_MAX,
                maxy=LAT_MAX,
                allow_one_dimensional_raster=True
            )
            
            # Save clipped raster
            output_filename = os.path.basename(tif_file)
            output_path = os.path.join(output_dir, output_filename)
            
            clipped.rio.to_raster(output_path, compress='LZW')
            
        except Exception as e:
            print(f"Error processing {tif_file}: {e}")
    
    print(f"\nClipped rasters saved to: {output_dir}")

# Run clipping (uncomment to execute)
# clip_rasters_to_bbox()

## 8. Verify Downloaded Data

In [ ]:
def verify_downloads():
    """Check what has been downloaded"""
    
    for data_type in DATA_TYPES:
        data_type_dir = os.path.join(ROOT_DIR, data_type)
        if os.path.exists(data_type_dir):
            tif_count = sum([len([f for f in files if f.endswith('.tif')]) 
                           for _, _, files in os.walk(data_type_dir)])
            print(f"{data_type}: {tif_count} .tif files")
        else:
            print(f"{data_type}: Directory not found")
    
    # Expected: ~744 files per data type (62 years × 12 months)
    print("\nExpected: ~744 files per data type (1960-2021, monthly)")

verify_downloads()

## 9. Test Clipping on One File

In [ ]:
# Test clipping on a single file
test_file = None

# Find first .tif file
for data_type in DATA_TYPES:
    data_type_dir = os.path.join(ROOT_DIR, data_type)
    if os.path.exists(data_type_dir):
        for root, dirs, files in os.walk(data_type_dir):
            for file in files:
                if file.endswith('.tif'):
                    test_file = os.path.join(root, file)
                    break
            if test_file:
                break
    if test_file:
        break

if test_file:
    print(f"Testing with: {test_file}")
    
    # Open and clip
    raster = rioxarray.open_rasterio(test_file, mask_and_scale=True)
    clipped = raster.rio.clip_box(
        minx=LON_MIN,
        miny=LAT_MIN,
        maxx=LON_MAX,
        maxy=LAT_MAX,
        allow_one_dimensional_raster=True
    )
    
    print(f"\nOriginal shape: {raster.shape}")
    print(f"Clipped shape: {clipped.shape}")
    print(f"\nClipped bounds:")
    print(f"  Lat: {clipped.y.min().values} to {clipped.y.max().values}")
    print(f"  Lon: {clipped.x.min().values} to {clipped.x.max().values}")
    
    # Plot
    clipped.plot()
else:
    print("No .tif files found yet. Run download first.")